In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

# 09. SFT Training Loop | 监督微调训练框架: 数据构造与 Loss Masking (SFT Training Loop)

什么是pre-train，什么是post-train？


### 预训练和微调

> **预训练 (Pre-training) vs 微调 (SFT，Supervised Fine-Tuning，监督微调)**
> * **预训练**：模型预测下一个 Token。给定一本书，每一个字都要算 Loss。
> * **SFT**：给定 `[Prompt] + [Response]`。我们**只关心**模型能不能输出正确的 `Response`。如果把 `Prompt` 也纳入 Loss 计算，模型就会去“背诵”人类的提问方式，而不是去“回答”问题。
> 
> **如何解决？（Loss Masking）**
> 在 PyTorch 的 `CrossEntropyLoss` 中，有一个神仙参数叫 `ignore_index`，默认值是 `-100`。我们只要把 `labels` 中属于 `Prompt` 和 `Padding` 的部分全部替换成 `-100`，这部分就不会产生任何梯度！


### CrossEntropyLoss 与 ignore_index 的数学原理

在 LLM 自回归训练中，`CrossEntropyLoss` 配合 `ignore_index=-100` 是实现“仅训练回答（Response）”的关键。下面从公式层面彻底讲清其数学机理。

#### 1. 标准交叉熵损失（单步）
对于某个位置 $i$，模型输出的 Logits 向量为 $\mathbf{z}_i \in \mathbb{R}^V$（$V$ 为词表大小），真实标签为 $y_i$（对应的 Token ID）。

该位置的损失 $\ell_i$ 定义为：
$$
\ell_i = -\log\left( \frac{e^{z_{i, y_i}}}{\sum_{j=1}^{V} e^{z_{i, j}}} \right) = -z_{i, y_i} + \log \sum_{j=1}^{V} e^{z_{i, j}}
$$

#### 2. 加入 ignore_index 的掩码机制
设 `ignore_index` 的值为 $m = -100$。我们定义一个**掩码指示变量** $M_i$：
$$
M_i = 
\begin{cases} 
1, & \text{if } y_i \neq -100 \\
0, & \text{if } y_i = -100
\end{cases}
$$

在 PyTorch 的默认实现（`reduction='mean'`）中，最终的 Loss $L$ 不是除以序列总长度 $N$，而是**仅除以有效 Token 的数量** $\sum M_i$：
$$
L = \frac{\sum_{i=1}^{N} M_i \cdot \ell_i}{\sum_{i=1}^{N} M_i}
$$

当 $y_i = -100$ 时，$M_i = 0$，该位置的损失被强制置为 **常数 0**（$M_i \cdot \ell_i = 0$），并且该位置**不参与分母的求和平均**。

#### 3. 梯度回传的完整推导（为什么梯度为零？）

##### 3.1 正常 Token（$y_i \neq -100$）的梯度
对于 Softmax + CrossEntropy，Loss 对 Logits 的偏导数为经典结论（预测概率 $p$ 与 One-hot 标签之差）：
$$
\frac{\partial \ell_i}{\partial \mathbf{z}_i} = \mathbf{p}_i - \mathbf{one\_hot}(y_i)
$$
其中 $\mathbf{p}_i = \text{Softmax}(\mathbf{z}_i)$。此梯度**非零**，继续通过链式法则传给模型权重 $W$（$\frac{\partial L}{\partial W} \neq 0$），权重得以更新。

##### 3.2 被忽略 Token（$y_i = -100$）的梯度
由于该位置的损失值被强制置为常数 0，即：
$$
\ell_i = 0
$$
对任意 Logits 分量 $z_{i, j}$ 求偏导，常数的导数为零
优化器执行参数更新时，其位置的参数**纹丝不动**。

#### 4. 训练应用（Prompt 与 Padding）

在 SFT 训练中，我们将序列划分为三部分：
- **Prompt（提示词）**：模型输入，不期望预测。
- **Response（回答）**：模型需要学习生成的目标。
- **Padding（填充位）**：用于 Batch 对齐的无意义占位符。

我们对 Labels（标签）进行处理：
$$
\text{Labels}[i] = 
\begin{cases} 
\text{Token ID}, & \text{if position } i \text{ belongs to Response} \\
-100, & \text{if position } i \text{ belongs to Prompt or Padding}
\end{cases}
$$

**数学结论**：
- **Response 部分**：$M_i = 1$，产生非零梯度，模型学习如何续写答案。
- **Prompt 与 Padding 部分**：$M_i = 0$，Loss 贡献为 0，梯度为 0，模型权重的更新完全忽略这些位置。


### 区分Pre-Train和Post-Train
#### Pre-train（预训练）
- **做什么**：在海量、杂乱的原始文本（无标签）上自学。
- **学什么**：学语法、常识、推理逻辑和世界知识。
- **结果**：变成一个基础“通才”模型（类似博览群书的高中生）。
- **成本**：极高（数千张GPU，数月时间）。

#### Post-train（后训练）
- **做什么**：在预训练基础上，用高质量、带标签的对话数据或偏好数据进一步训练。
- **学什么**：学指令遵循（听懂人话）、对齐人类价值观（有用/无害/诚实）。
- **结果**：变成一个能聊天、会做事的“助理”模型（类似上岗前的入职培训）。
- **常见手段**：SFT（监督微调）、RLHF（基于人类反馈的强化学习）、DPO（直接偏好优化）。

### Causal Masking 与 Shift Logits

在自回归语言模型中，预测第 $t+1$ 个词完全依赖于前 $t$ 个词。因此，在计算 CrossEntropyLoss 时，模型的预测输出序列（Logits）需要向左偏移（Shift）一位，与真实的标签序列（Labels）对齐。此外，对于 SFT 提示词部分，通常需要设置 `ignore_index = -100` 以避免它们产生梯度传播。

#### 为什么要 Shift？

> 根本原因：对齐错位
> 模型输入是 `[x1, x2, x3, x4]`，输出 Logits 与输入长度相同。

- `Logits[0]` 的职责是 **预测 `x2`**（基于 x1）。
- `Logits[1]` 的职责是 **预测 `x3`**（基于 x1,x2）。
- `Logits[2]` 的职责是 **预测 `x4`**（基于 x1,x2,x3）。

> 如果不 Shift（错误对齐）
> 直接用 `Logits` 对 `Labels` 计算 Loss：
- `Logits[0]`  vs `Labels[0]`（即 x1） ❌ （模型学的是用 x1 预测 x1，泄露标签）
- `Logits[1]`  vs `Labels[1]`（即 x2） ❌

> 如果 Shift（正确对齐）
> 取 `Logits[:, :-1]`（去掉最后一个） 对齐 `Labels[:, 1:]`（去掉第一个）：
- `Logits[0]`  vs `Labels[1]`（即 x2） ✅
- `Logits[1]`  vs `Labels[2]`（即 x3） ✅
- `Logits[2]`  vs `Labels[3]`（即 x4） ✅

> 一句话总结
> **因为模型在位置 `i` 输出的是对 `i+1` 的预测，Shift 是为了让“预测值”与“真实下一个词”在数学上对齐，避免模型作弊（直接用当前位置的词作为预测目标）。**

In [2]:
def build_sft_data(prompt_ids: list[int], response_ids: list[int], pad_id: int = 0, max_len: int = 16):
    """
    构造单条 SFT 训练数据
    """
    # 1. 拼接成完整的序列
    input_ids = prompt_ids + response_ids
    
    # ==========================================
    # TODO 1: 构造 labels
    # 规则：
    # - 长度与 input_ids 相同
    # - prompt 部分的 label 设置为 -100
    # - response 部分的 label 保持原样 (等于 input_ids 对应位置的值)
    # ==========================================
    labels = [-100] * len(prompt_ids) + response_ids
    
    # ==========================================
    # TODO 2: 截断 (Truncation) 与 填充 (Padding)
    # 规则：
    # - 如果超出 max_len，从末尾截断
    # - 如果不足 max_len，在末尾填充 (input_ids 填 pad_id，labels 填 -100)
    # ==========================================
    # 如果超长:
    if len(input_ids) > max_len:
        input_ids = input_ids[:max_len]
        labels = labels[:max_len]
    
    # 如果不足:
    else:
        pad_len = max_len - len(input_ids)
        input_ids = input_ids + [pad_id] * pad_len
        labels = labels + [-100] * pad_len
    
    return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

def compute_sft_loss(logits: torch.Tensor, labels: torch.Tensor):
    """
    计算自回归 SFT Loss
    Args:
        logits: [batch_size, seq_len, vocab_size]
        labels: [batch_size, seq_len]
    """
    # ==========================================
    # TODO 3: 实现 Shift 错位对齐
    # 将 logits 的最后一个 token 切掉
    # 将 labels 的第一个 token 切掉
    # ==========================================
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # ==========================================
    # TODO 4: 将 tensor 展平并计算交叉熵
    # ==========================================
    loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
    shift_logits = shift_logits.view(-1, shift_logits.size(-1))
    shift_labels = shift_labels.view(-1)
    loss = loss_fct(shift_logits, shift_labels)
    return loss


**工程要点**

- **内存效率**：使用 `ignore_index` 比手动 mask 更高效，因为 PyTorch 底层会跳过这些位置的计算。
- **梯度稳定性**：Shift 对齐确保每个位置的预测目标明确，避免了"预测自己"的混乱。
- **数据构造**：在实际工程中，通常在 DataLoader 中批量构造 labels，而不是逐条处理。